In [1]:
import os
from pathlib import Path
import torch
import torch.nn as nn 
from torch.nn import functional as F
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


max_iters = 100
eval_interval = 100
block_size = 8
batch_size = 4
learning_rate = 1e-2


In [2]:

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x,y


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_interval)
        for k in range(eval_interval):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # a lookup table for inputs
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, target=None):
        # idx :: batch_size(B)  *  block_size(T) 
        logits = self.token_embedding_table(idx) # B,T,C

        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, -1)
            target = target.view(-1)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens = 50):
        # Assuming ==> idx :: (B, T) : (1, T) 
        # i.e  Only a batch with input ==>  batch_size(B) = 1
        for _ in range(max_tokens):
            # get predictions
            logits, loss = self(idx)

            # get the last word/character predictions
            logits = logits[:, -1, :] # (B, 1, C)

            # get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # get the next token id by sampling from the distribution
            id = torch.multinomial(probs, num_samples=1) # (B, 1) <=> (1,1)

            # append sampled index to the running sequence
            idx = torch.cat((idx, id), dim=-1)
        return idx

def get_data():
    # data file path
    shakespeare_txt = Path(r"../data/raw/shakespeare.txt")

    if not os.path.exists(shakespeare_txt):
        raise FileNotFoundError(f"File not found: {shakespeare_txt}")

    if shakespeare_txt.exists():
        logger.info(f"file found : {shakespeare_txt.name}")
        logger.info(f"loading file : `{shakespeare_txt.name}` of size : {shakespeare_txt.stat().st_size / (1024)} KB")

        text = shakespeare_txt.read_text()
        logger.info(f"length of dataset in character : {len(text)}")
        logger.info(f"sample text : \n {text[:50]}")
    return text

def train(model, optimizer, max_iters=max_iters):
    for iter in range(max_iters):

        # every once in a while evaluate the loss on train and val sets
        if iter % eval_interval == 0:
            losses = estimate_loss()
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # get the batch
        xb, yb = get_batch('train')

        # evalute loss
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True) # make the previous gradients to none
        loss.backward() # calculate new gradients
        optimizer.step() # update the parameters

text = get_data()
char = sorted(list(set(text)))
vocab_size = len(char)

logger.info(f"vocab_size : {vocab_size}")
logger.info(f"unique characters : {str().join(char)}")

# create encoder and decoder for character level
stoi = {ch:i for i, ch in enumerate(char)}
itos = {i:ch for i, ch in enumerate(char)}

encoder = lambda s : [stoi[c] for c in s]
decoder = lambda l : "".join([itos[i] for i in l])

# create train and test dataset
data = torch.tensor(encoder(text), dtype=torch.long)

n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

2025-12-10 09:50:02,037 - __main__ - INFO - file found : shakespeare.txt
2025-12-10 09:50:02,038 - __main__ - INFO - loading file : `shakespeare.txt` of size : 306.9033203125 KB
2025-12-10 09:50:02,042 - __main__ - INFO - length of dataset in character : 306996
2025-12-10 09:50:02,043 - __main__ - INFO - sample text : 
 O for a Muse of fire, that would ascend
The bright
2025-12-10 09:50:02,045 - __main__ - INFO - vocab_size : 79
2025-12-10 09:50:02,045 - __main__ - INFO - unique characters : 
 !"'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz


In [3]:

model = BigramLanguageModel(vocab_size)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


In [4]:
train(model, optimizer, 10000)

step 0: train loss 4.7576, val loss 4.7500
step 100: train loss 4.1105, val loss 4.1082
step 200: train loss 3.6435, val loss 3.6459
step 300: train loss 3.2303, val loss 3.3020
step 400: train loss 2.9998, val loss 3.1265
step 500: train loss 2.8799, val loss 2.9095
step 600: train loss 2.7581, val loss 2.8100
step 700: train loss 2.7047, val loss 2.7137
step 800: train loss 2.6670, val loss 2.7216
step 900: train loss 2.6290, val loss 2.6834
step 1000: train loss 2.5746, val loss 2.6645
step 1100: train loss 2.5836, val loss 2.6524
step 1200: train loss 2.5610, val loss 2.6559
step 1300: train loss 2.5294, val loss 2.6659
step 1400: train loss 2.4920, val loss 2.6061
step 1500: train loss 2.5087, val loss 2.5821
step 1600: train loss 2.5083, val loss 2.5663
step 1700: train loss 2.4675, val loss 2.5705
step 1800: train loss 2.5094, val loss 2.5859
step 1900: train loss 2.5377, val loss 2.5783
step 2000: train loss 2.4187, val loss 2.5261
step 2100: train loss 2.4728, val loss 2.5961


In [5]:
# generate a sequence
context = torch.zeros((1,1), dtype=torch.long)

print(decoder(model.generate(context, 100).view(-1).tolist()))


 n wis,
 inees 'ct, my?   Catag de fak. bowfusts mmaisperof keng,  il'lowhepllm y at.  pogha t be n 
